# Prediction Accuracy & Bias Analysis
## LMArena Clean Summary Dataset (Bradley-Terry Predictions)

**Definitions (per LLM pair / group):**
- **Prediction:** Continuous value in [0, 1] from the Bradley-Terry model, representing P(LLM A wins)
- **Winner:** Binary ground truth — 1 if LLM A wins, 0 if LLM B wins
- **MSE (Brier Score):** `mean((winner - prediction)^2)` — lower is better
- **Bias:** `mean(prediction) - mean(winner)` — positive means the predictor over-predicts LLM A wins

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# -- Style (matches predictor_analysis.ipynb) --
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "axes.titleweight": "bold",
    "axes.labelweight": "semibold",
})

ACCENT  = "#E07A5F"
PRIMARY = "#3D405B"
CMAP    = "mako_r"

# -- Data source --
# Set this to a raw GitHub URL pointing to your clean_summary_v2.csv
DATA_URL = "https://raw.githubusercontent.com/<owner>/<repo>/<branch>/data/lmarena/clean_data/clean_summary_v2.csv"

OUT_DIR = pathlib.Path(".")
print("Data URL:", DATA_URL)

## 1. Load Data & Compute Per-Group Metrics

In [ ]:
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows, {df.group_id.nunique()} groups")
df.head()

In [ ]:
def compute_group_metrics(g):
    winner = g.winner.values
    pred = g.prediction.values
    n = len(g)

    mse = np.mean((winner - pred) ** 2)
    win_rate = winner.mean()
    pred_rate = pred.mean()
    bias = pred_rate - win_rate

    return pd.Series({
        "model_a": g.model_a.iloc[0],
        "model_b": g.model_b.iloc[0],
        "n": n,
        "mse": mse,
        "win_rate": win_rate,
        "pred_rate": pred_rate,
        "bias": bias,
    })

metrics = df.groupby("group_id").apply(compute_group_metrics).reset_index()
metrics["pair_label"] = metrics.model_a.str[:12] + " v " + metrics.model_b.str[:12]

# Ranked by descending win rate
metrics_ranked = metrics.sort_values("win_rate", ascending=False).reset_index(drop=True)
metrics_ranked["rank"] = np.arange(1, len(metrics_ranked) + 1)

metrics.describe()

## 2. Summary Statistics

In [ ]:
# Macro: mean across groups
macro_mse  = metrics.mse.mean()
macro_bias = metrics.bias.mean()

# Micro: pooled across all rows
micro_mse  = np.mean((df.winner.values - df.prediction.values) ** 2)
micro_bias = df.prediction.mean() - df.winner.mean()

print(f"Dataset: {len(df):,} rows, {len(metrics)} groups")
print(f"\nMicro (pooled)  — MSE: {micro_mse:.4f}   Bias: {micro_bias:+.4f}")
print(f"Macro (per-group) — MSE: {macro_mse:.4f}   Bias: {macro_bias:+.4f}")
print(f"\nGT overall win rate   : {df.winner.mean():.4f}")
print(f"Pred overall mean     : {df.prediction.mean():.4f}")

## 3. Plot 1 — MSE (Brier Score) Ranked by Descending GT Win Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_mse = plt.colormaps[CMAP](
    (metrics_ranked.win_rate - metrics_ranked.win_rate.min()) /
    (metrics_ranked.win_rate.max() - metrics_ranked.win_rate.min())
)
ax.bar(metrics_ranked["rank"], metrics_ranked.mse,
       color=colors_mse, width=1.0, edgecolor="none", alpha=0.85)
ax.axhline(macro_mse, color=ACCENT, ls="--", lw=1.2,
           label=f"Macro MSE = {macro_mse:.3f}")
ax.set_xlabel("LLM Pair Rank (by descending GT win rate of LLM A)")
ax.set_ylabel("MSE (Brier Score)")
ax.set_title("Prediction MSE Ranked by Descending Ground-Truth Win Rate")
ax.legend(fontsize=9)
ax.set_xlim(0.5, len(metrics_ranked) + 0.5)
ax.set_ylim(0, metrics_ranked.mse.max() * 1.1)
fig.tight_layout()
fig.savefig(OUT_DIR / "plot1_mse_ranked.png")
plt.show()

## 4. Plot 2 — Bias Ranked by Descending GT Win Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Color bars by sign: positive bias (over-predict) vs negative (under-predict)
bar_colors = np.where(metrics_ranked.bias >= 0, "#457B9D", "#E76F51")
ax.bar(metrics_ranked["rank"], metrics_ranked.bias,
       color=bar_colors, width=1.0, edgecolor="none", alpha=0.85)
ax.axhline(0, color="grey", ls="-", lw=0.8)
ax.axhline(macro_bias, color=ACCENT, ls="--", lw=1.2,
           label=f"Macro Bias = {macro_bias:+.3f}")
ax.set_xlabel("LLM Pair Rank (by descending GT win rate of LLM A)")
ax.set_ylabel("Bias (pred_rate $-$ win_rate)")
ax.set_title("Prediction Bias Ranked by Descending Ground-Truth Win Rate")
ax.legend(fontsize=9)
ax.set_xlim(0.5, len(metrics_ranked) + 0.5)
bias_lim = max(abs(metrics_ranked.bias.min()), abs(metrics_ranked.bias.max())) * 1.1
ax.set_ylim(-bias_lim, bias_lim)
fig.tight_layout()
fig.savefig(OUT_DIR / "plot2_bias_ranked.png")
plt.show()

## 5. Plot 3 — MSE & Bias vs GT Win Rate (LOWESS Trends)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# -- Left panel: MSE vs win rate --
ax = axes[0]
ax.scatter(metrics.win_rate, metrics.mse, s=28, alpha=0.45,
           color="#457B9D", edgecolors="white", linewidth=0.3)
sm = sm_lowess(metrics.mse.values, metrics.win_rate.values, frac=0.35)
ax.plot(sm[:, 0], sm[:, 1], color="#457B9D", lw=2.5, label="MSE (LOWESS)")
ax.axhline(macro_mse, color=ACCENT, ls="--", lw=1, label=f"Macro MSE = {macro_mse:.3f}")
ax.set_xlabel("Ground-Truth Win Rate (LLM A)")
ax.set_ylabel("MSE (Brier Score)")
ax.set_title("MSE vs Ground-Truth Win Rate")
ax.legend(fontsize=9, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(bottom=-0.01)

# -- Right panel: Bias vs win rate --
ax = axes[1]
ax.scatter(metrics.win_rate, metrics.bias, s=28, alpha=0.45,
           color="#E76F51", edgecolors="white", linewidth=0.3)
sm = sm_lowess(metrics.bias.values, metrics.win_rate.values, frac=0.35)
ax.plot(sm[:, 0], sm[:, 1], color="#E76F51", lw=2.5, label="Bias (LOWESS)")
ax.axhline(0, color="grey", ls="-", lw=0.8)
ax.axhline(macro_bias, color=ACCENT, ls="--", lw=1, label=f"Macro Bias = {macro_bias:+.3f}")
ax.set_xlabel("Ground-Truth Win Rate (LLM A)")
ax.set_ylabel("Bias (pred_rate $-$ win_rate)")
ax.set_title("Bias vs Ground-Truth Win Rate")
ax.legend(fontsize=9, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)

fig.tight_layout()
fig.savefig(OUT_DIR / "plot3_mse_bias_vs_winrate.png")
plt.show()

## 6. Plot 4 — Predicted vs Actual Win Rate Scatterplot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
sc = ax.scatter(
    metrics.win_rate, metrics.pred_rate,
    c=metrics.mse, cmap=CMAP, s=50, alpha=0.8,
    edgecolors="white", linewidth=0.4,
)
cbar = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.82)
cbar.set_label("MSE (Brier Score)", fontsize=10)

ax.plot([0, 1], [0, 1], color="grey", ls=":", lw=1.2, alpha=0.7, label="Perfect calibration")
ax.set_xlabel("Ground-Truth Win Rate (LLM A)")
ax.set_ylabel("Predicted Win Rate (mean prediction)")
ax.set_title("Calibration: Predicted vs Actual Win Rate")
ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(OUT_DIR / "plot4_calibration.png")
plt.show()

## 7. Plot 5 — Heatmap: MSE & Bias Binned by Win Rate

In [ ]:
metrics["wr_bin"] = pd.cut(metrics.win_rate, bins=np.linspace(0, 1, 11), include_lowest=True)
bin_agg = metrics.groupby("wr_bin", observed=False)[["mse", "bias"]].mean()

fig, ax = plt.subplots(figsize=(10, 3.2))
sns.heatmap(
    bin_agg.T, annot=True, fmt=".3f", cmap="RdBu_r", linewidths=0.5,
    ax=ax, center=0, cbar_kws={"shrink": 0.8, "label": "Value"},
)
ax.set_xlabel("Ground-Truth Win Rate Bin")
ax.set_ylabel("")
ax.set_title("Mean MSE & Bias by Win-Rate Bin")
ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()],
                   rotation=35, ha="right", fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "plot5_heatmap_by_winrate.png")
plt.show()

## 8. Interactive Exploration

Use the `metrics` DataFrame below to explore individual LLM pairs, filter by win rate, etc.

In [ ]:
# Top-10 pairs by lowest MSE (best calibrated)
metrics.nsmallest(10, "mse")[["model_a", "model_b", "n", "win_rate", "pred_rate", "mse", "bias"]]

In [ ]:
# Top-10 pairs by highest MSE (worst calibrated)
metrics.nlargest(10, "mse")[["model_a", "model_b", "n", "win_rate", "pred_rate", "mse", "bias"]]

In [ ]:
# Top-10 pairs by largest absolute bias
metrics.reindex(metrics.bias.abs().nlargest(10).index)[["model_a", "model_b", "n", "win_rate", "pred_rate", "mse", "bias"]]